In [8]:
!pip install langchain langchain-ollama langchain-community chromadb tiktoken


In [12]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings
from langchain_ollama import OllamaLLM
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

# --- 1. Load and split documents ---
loader = DirectoryLoader("data", loader_cls=TextLoader, glob="**/*.*")
docs = loader.load()

# Use smaller chunks to get more diverse retrieval
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
chunks = splitter.split_documents(docs)

# --- 2. Create embeddings + local vector store ---
embeddings = OllamaEmbeddings(model="nomic-embed-text")
vectorstore = Chroma.from_documents(chunks, embedding=embeddings, persist_directory="chroma_store")

# --- 3. Build retriever with diversity settings ---
retriever = vectorstore.as_retriever(
    search_type="mmr",  # Maximum Marginal Relevance for diversity
    search_kwargs={
        "k": 5,  # Retrieve more documents
        "lambda_mult": 0.7,  # Balance between relevance and diversity (0.0 = max diversity, 1.0 = max relevance)
        "fetch_k": 10  # Fetch more candidates before MMR selection
    }
)
llm = OllamaLLM(model="deepseek-coder:1.3b")

# --- 4. Create a custom prompt template with system instructions ---
prompt_template = """You are a Python programming assistant. Use the following code context to answer the user's technical question.
If you need to provide code, use the patterns and functions from the context as a foundation.

Context:
{context}

Question: {question}

Answer with complete, working Python code based on the context:"""

PROMPT = PromptTemplate(
    template=prompt_template, 
    input_variables=["context", "question"]
)

# --- 5. Combine into a RAG chain with custom prompt ---
qa = RetrievalQA.from_chain_type(
    llm=llm, 
    retriever=retriever, 
    chain_type="stuff", 
    return_source_documents=True,
    chain_type_kwargs={"prompt": PROMPT}
)

# --- 6. Ask a technical question ---
query = """Create a complete Python script that shows how to use the sales analysis pipeline. 
I need to process sales data from CSV files, run the analysis, and generate a summary report. 
Show me the main execution code that ties all the modules together."""

result = qa({"query": query})

print("🧠 Answer:\n", result["result"], "\n")
print("📚 Retrieved from:")
for doc in result["source_documents"]:
    print(" -", doc.metadata["source"])

🧠 Answer:
 Below is an example of how you can use this pipeline to process sales data from CSV files in your project directory (assuming each file has a column named 'date' for date and another one called SalesQuantity). This script will load all csv Files, run full analysis on them by using `SalesAnalyzer` class which contains functions like mean(), median() etc., then generate charts based upon the results.
Please note that you have to ensure your project directory has these files: data_loader.py - module for Data Loading; sales_analyzer.py – The Sales Analyse Module, chart_generator.py -- Chart Generator and report_genaror.py—Report Geneartor modules as they are defined in the context of provided code snippets but not present here because these files have to be implemented individually based on your project structure or file requirements/structure within them respectively;

```python 
import os    # To access and manage directories  
from data_loader import load_sales_data     # Loa

In [13]:
# Let's try a more specific question that should pull from multiple files
query2 = """I need to understand how to create visualizations for sales data. 
Show me the chart generation functions and explain how to create a dashboard with multiple plots."""

result2 = qa({"query": query2})

print("🎨 Visualization Answer:\n", result2["result"], "\n")
print("📚 Retrieved from:")
for doc in result2["source_documents"]:
    print(" -", doc.metadata["source"])

print("\n" + "="*50 + "\n")

# And another query for reports
query3 = """How do I generate executive summaries and detailed reports from analysis results? 
Show me the report generation code."""

result3 = qa({"query": query3})

print("📊 Report Generation Answer:\n", result3["result"], "\n")
print("📚 Retrieved from:")
for doc in result3["source_documents"]:
    print(" -", doc.metadata["source"])

🎨 Visualization Answer:
 Sure! We'll start by creating `ChartGenerator` class that will generate all charts needed in our visualizations using matplotlib library (seaborn style). Then we can create a method for generating comprehensive sales dashboard. Finally to export these plots, I would suggest you use the function from PIL or similar libraries because saving figures as images has some advantages like better support of different formats and scalability etc in general:

Here is an example code that meets your requirements using `matplotlib` library for generating visualizations (seaborn style) along with Python's standard logging module. This assumes you already have data related to sales which can be passed into the method as parameters, such a function could look like this - 
```python
import matplotlib.pyplot as plt   # We will use pyplot from here onwards for convenience in plotting graphs and subplots etc..
from os import makedirs           # For creating directories if they do